# Research notebook

Read the module README before executing. Textual outputs below are saved historical snapshots; the report follows the original project document and was not recalculated during publication cleanup. Setup paths have been made portable. Embedded media and machine-specific diagnostic output are omitted from this public-facing copy.


In [ ]:
# Portable project paths; this cell does not load a model.
from pathlib import Path
import sys

_candidates = (Path.cwd(), *Path.cwd().parents)
_project_root = next((p for p in _candidates if (p / "project_paths.py").is_file()), None)
if _project_root is None:
    raise RuntimeError("Start Jupyter from the retrieval repository or one of its subdirectories.")
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
from project_paths import PICTURE_DIR, GALLERY_DIR, OUTPUT_DIR, LLAVA_MODEL, TARGET_CLASSES

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [1]:

import os
import torch
from llava.constants import (
    IMAGE_TOKEN_INDEX,
    DEFAULT_IMAGE_TOKEN,
    DEFAULT_IM_START_TOKEN,
    DEFAULT_IM_END_TOKEN,
    IMAGE_PLACEHOLDER,
)
from llava.conversation import conv_templates, SeparatorStyle
from llava.model.builder import load_pretrained_model
from llava.utils import disable_torch_init
from llava.mm_utils import (
    process_images,
    tokenizer_image_token,
    get_model_name_from_path,
)
from io import BytesIO
import requests
import re
from PIL import Image
# ----------------------
# 1. 模型加载与初始化
# ----------------------
class LLaVABatchInference:
    def __init__(self, model_path=LLAVA_MODEL, load_8bit=True):
        self.model_path = model_path
        self.load_8bit = load_8bit
        self._load_model()

    def _load_model(self):
        disable_torch_init()
        model_name = get_model_name_from_path(self.model_path)
        self.tokenizer, self.model, self.image_processor, self.context_len = load_pretrained_model(
            self.model_path,
            model_base=None,
            model_name=model_name,
            load_8bit=self.load_8bit,
            device=torch.device("cuda:0")
        )
        self.model.eval()
        print(f"模型加载完成：{model_name}")

    def _get_conv_template(self, model_name):
        if "llama-2" in model_name.lower():
            return "llava_llama_2"
        elif "mistral" in model_name.lower():
            return "mistral_instruct"
        elif "v1.5" in model_name.lower():
            return "llava_v1"
        else:
            return "llava_v0"

# ----------------------
# 2. 图像处理与推理函数
# ----------------------
def load_image(image_path_or_url):
    """Resolve relative candidate paths against the configured gallery root."""
    image_path_or_url = str(image_path_or_url)
    if image_path_or_url.startswith(("http", "https")):
        response = requests.get(image_path_or_url)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image_path = Path(image_path_or_url).expanduser()
        if not image_path.is_absolute():
            image_path = PICTURE_DIR / image_path
        image = Image.open(image_path).convert("RGB")
    return image

def generate_prompt(query, model, use_im_start_end=True):
    """构建带图像标记的 Prompt"""
    if use_im_start_end and hasattr(model.config, "mm_use_im_start_end") and model.config.mm_use_im_start_end:
        image_token = f"{DEFAULT_IM_START_TOKEN}{DEFAULT_IMAGE_TOKEN}{DEFAULT_IM_END_TOKEN}"
    else:
        image_token = DEFAULT_IMAGE_TOKEN
    return f"{image_token}\n{query}"

def single_image_inference(model_obj, image_path, query, temperature=0.2):
    """单图像推理函数（返回文本结果）"""
    # 1. 加载并预处理图像
    image = load_image(image_path)
    images_tensor = process_images([image], model_obj.image_processor, model_obj.model.config).to(
        model_obj.model.device, dtype=torch.float16
    )
    image_size = [image.size]  # 模型需要的图像尺寸信息

    # 2. 构建 Prompt
    prompt = generate_prompt(query, model_obj.model)
    conv_mode = model_obj._get_conv_template(get_model_name_from_path(model_obj.model_path))
    conv = conv_templates[conv_mode].copy()
    conv.append_message(conv.roles[0], prompt)
    conv.append_message(conv.roles[1], None)
    formatted_prompt = conv.get_prompt()

    # 3. 编码输入
    input_ids = tokenizer_image_token(
        formatted_prompt, model_obj.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt"
    ).unsqueeze(0).to(model_obj.model.device)

    # 4. 生成回答
    with torch.inference_mode():
        output_ids = model_obj.model.generate(
            input_ids,
            images=images_tensor,
            image_sizes=image_size,
            do_sample=(temperature > 0),
            temperature=temperature,
            max_new_tokens=512,
            use_cache=True
        )
    output = model_obj.tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
    return {
        # Keep the original identifier so answers still match the manifest.
        "Path": image_path,
        "Pre": output
    }



In [2]:
MODEL_PATH = LLAVA_MODEL
LOAD_8BIT = False
model_obj = LLaVABatchInference(
    model_path=MODEL_PATH,
    load_8bit=LOAD_8BIT
)
TEMPERATURE = 0.5

模型加载完成：llava-v1.5-7b


In [4]:
picture = list(TARGET_CLASSES)

In [5]:
import pickle
for figure in picture:
    print(figure)
    with open(OUTPUT_DIR / f'{figure} llava dataset', 'rb') as f:
        dataset=pickle.load(f)
    indices = [i for i, value in enumerate(dataset['Pre']) if value == 1]
    image_paths=[dataset['Path'][i] for i in indices]
    print(len(image_paths),len(indices))
    QUERY = f"Is this an image of a {figure}? Please answer with Yes or No."
    if not image_paths:
         print("警告：未找到有效图像文件")
    else:
    # 执行批量推理
        results = {'Path':[],'Pre':[]}
        for idx, image_path in enumerate(image_paths):
            print(f"处理图像 {idx+1}/{len(image_paths)}: {image_path}")
            try:
                result = single_image_inference(
                model_obj=model_obj,
                image_path=image_path,
                query=QUERY,
                temperature=TEMPERATURE
                )
                results['Path'].append(result['Path'])
                results['Pre'].append(result['Pre'])
                print(result)
            except Exception as e:
                print(f"错误：{image_path} 处理失败 - {str(e)}")
                continue
    with open(OUTPUT_DIR / f'{figure}推理结果', 'wb') as f:
        pickle.dump(results, f)

Dog
227 227
Piano
323 323


Erhu
235 235


Porcelain
301 301
Duck
280 280


In [6]:
picture = list(TARGET_CLASSES)
evaluation={x:{'Precision':0,'Recall':0,'F1':0} for x in picture}
for figure in picture:
    with open(OUTPUT_DIR / f'{figure}推理结果', 'rb') as f:
        newdata=pickle.load(f)
    with open(OUTPUT_DIR / f'{figure} llava dataset', 'rb') as f:
        data0=pickle.load(f)
    data=data0.copy()
    for x_path,x_pred in zip(newdata['Path'], newdata['Pre']):
        index=data['Path'].index(x_path)
        if 'No'in x_pred:
            print(index)
            data['Pre'][index]=0
    T=data['True']
    P=data['Pre']
    TP=sum(1 for i,j in zip(T,P) if i==1 and j==1)
    FP=sum(1 for i,j in zip(T,P) if i==0 and j==1)
    FN=sum(1 for i,j in zip(T,P) if i==1 and j==0)
    precision=TP/(TP+FP)
    recall=TP/(TP+FN)
    evaluation[figure]['Precision']=precision
    evaluation[figure]['Recall']=recall
    evaluation[figure]['F1']=2*precision*recall/(precision+recall)






10
15
19
24
62
69
100
119
123
127
128
136
156
187
196
252
255
256
264
269
274
294
353
361
363
366
372
375
382
385
390
393
396
397
398
507
1421
1603
1956
1300
1322
1343
1346
1407
1409
1420
1421
1425
1430
1442
1443
1471
1478
1492
1530
1544
7
36
152
1051
1066
1093
1094
1118
1123
1148
1157
1159
1173
1192
160
165
179
193
195
510
1401
1442
1557
1571
1644
1659
1761
1774
1777
1780
1784
1789
1793
1794
1803
1806
1807
1809
1821
1832
1833
1834
1854
1868
1874
1877
1879
1881
1883
1884
1888
1905
1906
1913
1923
1927
1939
1940
1941
1942
1949
1957
1964
1976
1979
7
72
129
136
600
605
606
607
608
612
613
614
615
631
636
653
657
659
674
678
680
683
687
691
694
701
707
715
719
731
735
745
749
779
1918


In [7]:

print(evaluation)

{'Dog': {'Precision': 0.9627659574468085, 'Recall': 0.905, 'F1': 0.9329896907216496}, 'Piano': {'Precision': 0.6372549019607843, 'Recall': 0.975, 'F1': 0.7707509881422924}, 'Erhu': {'Precision': 0.8461538461538461, 'Recall': 0.935, 'F1': 0.8883610451306413}, 'Porcelain': {'Precision': 0.776, 'Recall': 0.97, 'F1': 0.8622222222222223}, 'Duck': {'Precision': 0.7918367346938775, 'Recall': 0.97, 'F1': 0.8719101123595505}}
